# Multimodal J-lens capture on Gemma 4 E4B — first image- and audio-conditioned records

Produces the explorer's first **real** image-conditioned and audio-conditioned
records: for one user-supplied image and one user-supplied audio clip, capture
the decoder residual at layer 38 (position −1, the final text-generation
position), read it out through the **frozen, text-fitted** pilot lens, and
decompose it with the same k=10 gradient pursuit as the completed text run.

**Interpretation boundary, recorded in every artifact:** the lens was fitted
on 100 *text* prompts. Applying it to multimodal-conditioned decoder states is
an exploratory probe of those states — it does **not** establish that text and
multimodal inputs share concepts, and nothing here claims pixel- or
audio-span-level attribution. Modality token ranges are recorded only when the
processor exposes them.

Inputs are configured in the **INPUTS** cell below (nothing is committed to
the repository). Safe end-to-end on an L4 (≈15–25 min including model load).
Outside Colab every model-touching cell is a gated no-op (light path).

## 0. Colab setup (skip if running locally)

In [ ]:
# 0. Colab bootstrap: clone/update the private repo from a fresh runtime.
# No-op outside Colab. Never loads Gemma; only touches git/pip.
import base64
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if not IN_COLAB:
    print("Not running in Colab — skipping bootstrap; using the local checkout.")
else:
    CHECKOUT_DIR = Path("/content/jacobian-lens-gemma")
    REPO_URL = "https://github.com/MechInterpreter/jacobian-lens-gemma.git"
    BRANCH = "multimodal-jlens-explorer"

    def _normalize(url: str) -> str:
        return url.strip().removesuffix(".git").removesuffix("/")

    try:
        from google.colab import userdata
        GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
    except Exception:
        GITHUB_TOKEN = None
    if not GITHUB_TOKEN:
        raise RuntimeError(
            "GITHUB_TOKEN secret not found or not accessible. In Colab: open "
            "the key icon (Secrets) in the left sidebar, add a secret named "
            "GITHUB_TOKEN containing a fine-grained GitHub token with "
            "read-only access to MechInterpreter/jacobian-lens-gemma, and "
            "enable notebook access for it. Then re-run this cell."
        )

    # Auth via a per-invocation extraHeader override: lives only in argv,
    # never written to .git/config, the remote URL, or notebook output.
    _token_b64 = base64.b64encode(f"x-access-token:{GITHUB_TOKEN}".encode()).decode()
    _auth_header = f"AUTHORIZATION: basic {_token_b64}"
    _auth_args = ["-c", f"http.https://github.com/.extraHeader={_auth_header}"]

    def _run(args, *, auth=False, check=True):
        cmd = ["git", *(_auth_args if auth else []), *args]
        result = subprocess.run(cmd, capture_output=True, text=True)
        if check and result.returncode != 0:
            safe_stderr = result.stderr.replace(GITHUB_TOKEN, "***")
            raise RuntimeError(f"git {' '.join(args)} failed:\n{safe_stderr}")
        return result.stdout.strip()

    if not CHECKOUT_DIR.exists():
        print(f"Cloning {REPO_URL} ({BRANCH}) into {CHECKOUT_DIR} ...")
        _run(["clone", "--branch", BRANCH, REPO_URL, str(CHECKOUT_DIR)], auth=True)
    else:
        if not (CHECKOUT_DIR / ".git").exists():
            raise RuntimeError(
                f"{CHECKOUT_DIR} exists but is not a git checkout; refusing to "
                "touch it. Remove or rename it manually, then re-run this cell."
            )
        existing_url = _run(["-C", str(CHECKOUT_DIR), "remote", "get-url", "origin"])
        if _normalize(existing_url) != _normalize(REPO_URL):
            raise RuntimeError(
                f"{CHECKOUT_DIR} is checked out from {existing_url!r}, not "
                f"{REPO_URL!r}; refusing to touch an unexpected repository. "
                "Remove or rename it manually, then re-run this cell."
            )
        print(f"Updating existing checkout at {CHECKOUT_DIR} ...")
        _run(["-C", str(CHECKOUT_DIR), "fetch", "origin"], auth=True)
        _run(["-C", str(CHECKOUT_DIR), "checkout", BRANCH])
        _run(["-C", str(CHECKOUT_DIR), "merge", "--ff-only", f"origin/{BRANCH}"])

    final_url = _run(["-C", str(CHECKOUT_DIR), "remote", "get-url", "origin"])
    if "@" in final_url or GITHUB_TOKEN in final_url:
        raise RuntimeError("origin URL unexpectedly contains credentials; aborting.")

    os.chdir(CHECKOUT_DIR)
    if str(CHECKOUT_DIR) not in sys.path:
        sys.path.insert(0, str(CHECKOUT_DIR))

    print("Installing the 'gemma' extra ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", ".[gemma]"],
        check=True,
    )

    branch_now = _run(["-C", str(CHECKOUT_DIR), "rev-parse", "--abbrev-ref", "HEAD"])
    sha_now = _run(["-C", str(CHECKOUT_DIR), "rev-parse", "HEAD"])
    print(f"checked out: {branch_now} @ {sha_now}")
    assert (CHECKOUT_DIR / "jlens").is_dir(), f"{CHECKOUT_DIR}/jlens not found"


In [ ]:
# 1. Execution gates. Model loading and CUDA are opt-in; outside Colab the
# defaults keep everything in the light (no-download) path.
import os
import sys

IN_COLAB = "google.colab" in sys.modules
os.environ.setdefault("JLENS_ALLOW_GEMMA", "1" if IN_COLAB else "0")
os.environ.setdefault("JLENS_DEVICE_MAP", "cuda" if IN_COLAB else "")
ALLOW_MODEL_LOAD = os.environ.get("JLENS_ALLOW_GEMMA", "0") == "1"
DEVICE_MAP = os.environ.get("JLENS_DEVICE_MAP") or None
print(f"IN_COLAB={IN_COLAB}  ALLOW_MODEL_LOAD={ALLOW_MODEL_LOAD}  DEVICE_MAP={DEVICE_MAP}")


In [ ]:
# 2. INPUTS — set your image and audio here (paths on the Colab VM or
# Drive). Environment variables override the config; the resolved values are
# recorded in resolved_config.json. Do not commit sample assets to the repo.
IMAGE_PATH = os.environ.get("IMAGE_PATH") or None    # e.g. "/content/drive/MyDrive/jacobian-lens-gemma/inputs/photo.jpg"
IMAGE_PROMPT = os.environ.get("IMAGE_PROMPT") or None  # None -> config default
AUDIO_PATH = os.environ.get("AUDIO_PATH") or None    # e.g. "/content/drive/MyDrive/jacobian-lens-gemma/inputs/clip.wav"
AUDIO_PROMPT = os.environ.get("AUDIO_PROMPT") or None  # None -> config default
print(f"IMAGE_PATH={IMAGE_PATH}\nAUDIO_PATH={AUDIO_PATH}")


In [ ]:
# 3. Environment and provenance (no model load).
import json
import pathlib

import torch

from jlens.metadata import environment_manifest

ENV = environment_manifest()
print(json.dumps(ENV, indent=2))


## Persisting outputs to Google Drive

A fresh timestamped run directory under `MyDrive/jacobian-lens-gemma/runs/`;
each modality's record is written as soon as it is captured.

In [ ]:
# 4. Google Drive persistence (Colab only). No-op outside Colab.
from pathlib import Path

if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
    except Exception as exc:
        raise RuntimeError(
            f"failed to mount Google Drive at /content/drive: {exc}. Approve "
            "the Drive authorization prompt when it appears, then re-run this cell."
        ) from exc

    DRIVE_MOUNT = Path("/content/drive")
    if not DRIVE_MOUNT.is_dir():
        raise RuntimeError("Drive did not mount successfully.")

    PERSIST_ROOT = DRIVE_MOUNT / "MyDrive" / "jacobian-lens-gemma"
    RUNS_ROOT = PERSIST_ROOT / "runs"
    RUNS_ROOT.mkdir(parents=True, exist_ok=True)
    print(f"RUNS_ROOT = {RUNS_ROOT}")
else:
    PERSIST_ROOT = None
    RUNS_ROOT = Path("runs")
    print("Not in Colab — using the local runs/ directory (read) and "
          "artifacts/multimodal_capture (write).")


In [ ]:
# 5. Load and validate the capture configuration; create the run directory
# (fresh-run overwrite protection) and record the resolved inputs.
from datetime import datetime, timezone

from jlens.metadata import (
    config_fingerprint,
    execution_record,
    load_multimodal_config,
    write_metadata,
)

CONFIG_PATH = "configs/gemma_multimodal_jlens_capture.yaml"
CONFIG = load_multimodal_config(CONFIG_PATH)
FINGERPRINT = config_fingerprint(CONFIG)
CAP = CONFIG["capture"]

IMAGE_PATH = IMAGE_PATH or CONFIG["inputs"]["image_path"]
AUDIO_PATH = AUDIO_PATH or CONFIG["inputs"]["audio_path"]
IMAGE_PROMPT = IMAGE_PROMPT or CONFIG["inputs"]["image_prompt"]
AUDIO_PROMPT = AUDIO_PROMPT or CONFIG["inputs"]["audio_prompt"]
if not IMAGE_PATH and not AUDIO_PATH:
    print("WARNING: neither IMAGE_PATH nor AUDIO_PATH is set — the capture "
          "cells will be no-ops. Set them in the INPUTS cell above.")

EXECUTION = execution_record(
    configured_allow_model_load=CONFIG["model"]["allow_model_load"],
    resolved_allow_model_load=ALLOW_MODEL_LOAD,
    model_loaded=False,
    override_source="notebook:JLENS_ALLOW_GEMMA" if ALLOW_MODEL_LOAD else None,
)

if IN_COLAB:
    RUN_ID = (f"multimodal_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%f')}_"
              f"{FINGERPRINT.removeprefix('sha256:')[:12]}")
    RUN_DIR = RUNS_ROOT / RUN_ID
    RUN_DIR.mkdir(parents=True, exist_ok=False)  # fresh-run overwrite protection
    OUTPUT_DIR = RUN_DIR / "artifacts"
else:
    RUN_ID = None
    RUN_DIR = None
    OUTPUT_DIR = pathlib.Path(CONFIG["paths"]["output_dir"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "assets").mkdir(exist_ok=True)

if RUN_DIR is not None:
    write_metadata(str(RUN_DIR / "run_started.json"), {
        "run_id": RUN_ID,
        "mode": "multimodal_capture",
        "config_fingerprint": FINGERPRINT,
        "execution": EXECUTION,
    })
    write_metadata(str(RUN_DIR / "resolved_config.json"), {
        "config": CONFIG,
        "config_fingerprint": FINGERPRINT,
        "execution": EXECUTION,
        "resolved_inputs": {
            "image_path": IMAGE_PATH, "image_prompt": IMAGE_PROMPT,
            "audio_path": AUDIO_PATH, "audio_prompt": AUDIO_PROMPT,
        },
    })
print(f"config: {CONFIG_PATH}\nfingerprint: {FINGERPRINT}")
print(f"RUN_DIR = {RUN_DIR}\nOUTPUT_DIR = {OUTPUT_DIR}")


In [ ]:
# 6. Lightweight validation suite (CPU, mocks, no network) — must pass
# before any real-model work.
import subprocess

proc = subprocess.run(
    [sys.executable, "-m", "pytest", "tests", "-q", "--no-header"],
    capture_output=True, text=True,
)
print(proc.stdout[-3000:])
if proc.returncode != 0:
    print(proc.stderr[-3000:])
    raise RuntimeError("local test suite failed; fix before running the model")


In [ ]:
# 7. Locate and verify the FROZEN pilot lens. Read-only; never refitted.
from jlens.lens import JacobianLens
from jlens.metadata import file_sha256

LENS_CFG = CONFIG["lens"]
PILOT_RUN_DIR = RUNS_ROOT / LENS_CFG["run_dir_name"]
LENS_PATH = PILOT_RUN_DIR / LENS_CFG["artifact_relpath"]
if not LENS_PATH.is_file():
    raise RuntimeError(
        f"pilot lens not found at {LENS_PATH}. In Colab, RUNS_ROOT must be the "
        "Drive runs/ directory containing the completed pilot run "
        f"{LENS_CFG['run_dir_name']!r}."
    )

lens_sha = file_sha256(str(LENS_PATH))
print(f"lens file: {LENS_PATH}\nsha256:    {lens_sha}")
if LENS_CFG["expect_file_sha256"] and lens_sha != LENS_CFG["expect_file_sha256"]:
    raise RuntimeError(
        f"lens fingerprint mismatch: expected {LENS_CFG['expect_file_sha256']}, "
        f"got {lens_sha} — refusing to capture against an unexpected lens"
    )

LENS = JacobianLens.load(str(LENS_PATH))
assert LENS.source_layers == LENS_CFG["expect_source_layers"], LENS.source_layers
assert LENS.n_prompts == LENS_CFG["expect_n_prompts"], LENS.n_prompts
assert LENS.d_model == CONFIG["model"]["expect_d_model"], LENS.d_model
for capture_layer in CAP["layers"]:
    assert capture_layer in LENS.jacobians, f"layer {capture_layer} not fitted"
for layer, J in LENS.jacobians.items():
    assert torch.isfinite(J).all(), f"non-finite J at layer {layer}"

pilot_meta_path = PILOT_RUN_DIR / "run_metadata.json"
PILOT_META = json.load(open(pilot_meta_path, encoding="utf-8"))
LENS_VERIFICATION = {
    "lens_path": str(LENS_PATH),
    "file_sha256": lens_sha,
    "source_layers": LENS.source_layers,
    "n_prompts": LENS.n_prompts,
    "d_model": LENS.d_model,
    "all_finite": True,
    "pilot_run_id": PILOT_META["run_id"],
    "pilot_model_revision": PILOT_META["model_revision"],
    "pilot_config_fingerprint": PILOT_META["config_fingerprint"],
}
print(json.dumps(LENS_VERIFICATION, indent=2))


In [ ]:
# 8. Load the model AND the multimodal processor (gated), then resolve the
# processor/model interface by inspection — never by assumption. The resolved
# interface is printed and recorded; unsupported modalities fail clearly.
import inspect

from jlens.gemma4 import load_gemma4, verify_architecture

MODEL = None
LOAD_INFO = None
ARCH_REPORT = None
PROCESSOR = None
INTERFACE = None
if not ALLOW_MODEL_LOAD:
    print("JLENS_ALLOW_GEMMA != 1 — light path: skipping model/processor load.")
else:
    MODEL, LOAD_INFO = load_gemma4(
        CONFIG["model"]["repo_id"],
        revision=CONFIG["model"]["revision"],
        dtype=getattr(torch, CONFIG["model"]["dtype"]),
        device_map=DEVICE_MAP,
        allow_model_load=True,
    )
    for parameter in MODEL._hf_model.parameters():
        parameter.requires_grad_(False)
    ARCH_REPORT = verify_architecture(
        MODEL,
        expect_n_layers=CONFIG["model"]["expect_n_layers"],
        expect_d_model=CONFIG["model"]["expect_d_model"],
        expect_vocab_size=CONFIG["model"]["expect_vocab_size"],
    ).to_dict()
    EXECUTION = execution_record(
        configured_allow_model_load=CONFIG["model"]["allow_model_load"],
        resolved_allow_model_load=True,
        model_loaded=True,
        override_source="notebook:JLENS_ALLOW_GEMMA",
    )

    from transformers import AutoProcessor
    PROCESSOR = AutoProcessor.from_pretrained(
        CONFIG["model"]["repo_id"], revision=LOAD_INFO["model_revision"]
    )
    call_params = list(inspect.signature(PROCESSOR.__call__).parameters)
    audio_kwarg = next((k for k in ("audio", "audios", "raw_audio") if k in call_params), None)
    hf_config = MODEL._hf_model.config
    INTERFACE = {
        "processor_class": type(PROCESSOR).__name__,
        "components": {
            name: type(getattr(PROCESSOR, name)).__name__
            for name in ("tokenizer", "image_processor", "feature_extractor",
                         "audio_processor", "video_processor")
            if getattr(PROCESSOR, name, None) is not None
        },
        "call_parameters": call_params,
        "supports_image": getattr(PROCESSOR, "image_processor", None) is not None
                          and "images" in call_params,
        "supports_audio": audio_kwarg is not None and any(
            getattr(PROCESSOR, name, None) is not None
            for name in ("feature_extractor", "audio_processor")),
        "audio_kwarg": audio_kwarg,
        "image_token_id": next((getattr(hf_config, name) for name in
                                ("image_token_index", "image_token_id")
                                if getattr(hf_config, name, None) is not None), None),
        "audio_token_id": next((getattr(hf_config, name) for name in
                                ("audio_token_index", "audio_token_id")
                                if getattr(hf_config, name, None) is not None), None),
        "has_chat_template": callable(getattr(PROCESSOR, "apply_chat_template", None)),
    }
    print("resolved processor/model interface:")
    print(json.dumps(INTERFACE, indent=2, default=str))
    if IMAGE_PATH and not INTERFACE["supports_image"]:
        raise RuntimeError(
            "IMAGE_PATH is set but the resolved processor does not accept "
            f"images (call parameters: {call_params}); refusing to fake it")
    if AUDIO_PATH and not INTERFACE["supports_audio"]:
        raise RuntimeError(
            "AUDIO_PATH is set but the resolved processor does not accept "
            f"audio (call parameters: {call_params}). This checkpoint/processor "
            "does not expose an audio pathway — audio capture is unsupported "
            "here, and no fake data will be substituted")


In [ ]:
# 9. Capture machinery: build inputs, run one forward with the recorder at
# the capture layers, read out the lens, run the k=10 pursuit, and assemble
# schema-shaped records. Everything recorded, nothing assumed.
import hashlib
import re
import shutil

from jlens.explorer_export import (
    _cone_to_bundle_record,   # bundle-shape converters (module-internal names,
    _cone_to_pursuit_trace,   # reused deliberately so shapes match the exporter)
    example_id as make_example_id,
)
from jlens.cones import make_cone_record
from jlens.hooks import ActivationRecorder
from jlens.pursuit import JSpaceDictionary, PursuitSettings, gradient_pursuit

DEC = CONFIG["decomposition"]
SETTINGS = PursuitSettings(
    k=CAP["k"],
    normalize_atoms=DEC["normalize_atoms"],
    refine_steps=DEC["refine_steps"],
    tol_relative_residual=float(DEC["tol_relative_residual"]),
    correlation_chunk_size=DEC["correlation_chunk_size"],
)

def _hash16(text):
    return hashlib.sha256(text.encode()).hexdigest()[:16]

def _slugify(stem):
    return re.sub(r"[^a-z0-9-]+", "-", stem.lower()).strip("-") or "input"

def _token_range(input_ids, token_id):
    """[start, end) of a contiguous run of token_id, else None (recorded,
    not guessed)."""
    if token_id is None:
        return None
    hits = (input_ids[0] == int(token_id)).nonzero().flatten().tolist()
    if not hits:
        return None
    if hits[-1] - hits[0] + 1 != len(hits):
        print(f"token id {token_id} occurs non-contiguously; recording no range")
        return None
    return [int(hits[0]), int(hits[-1]) + 1]

def build_inputs(prompt, *, image=None, audio=None, sampling_rate=None):
    """Try the direct processor call first; fall back to the chat template.
    Returns (inputs, route_metadata). Never silently substitutes data."""
    kwargs = {"text": prompt, "return_tensors": "pt"}
    if image is not None:
        kwargs["images"] = image
    if audio is not None:
        kwargs[INTERFACE["audio_kwarg"]] = audio
        if sampling_rate is not None and "sampling_rate" in INTERFACE["call_parameters"]:
            kwargs["sampling_rate"] = sampling_rate
    try:
        return PROCESSOR(**kwargs), {"route": "processor_call",
                                     "kwargs": sorted(k for k in kwargs if k != "text")}
    except Exception as exc:
        if not INTERFACE["has_chat_template"]:
            raise
        content = []
        if image is not None:
            content.append({"type": "image", "image": image})
        if audio is not None:
            content.append({"type": "audio", "audio": audio})
        content.append({"type": "text", "text": prompt})
        inputs = PROCESSOR.apply_chat_template(
            [{"role": "user", "content": content}],
            add_generation_prompt=True, tokenize=True,
            return_dict=True, return_tensors="pt",
        )
        return inputs, {"route": "chat_template",
                        "direct_call_error": str(exc)[:300]}

@torch.no_grad()
def capture(prompt, *, modality, image=None, audio=None, sampling_rate=None,
            asset_name=None, asset_meta=None):
    inputs, route = build_inputs(prompt, image=image, audio=audio,
                                 sampling_rate=sampling_rate)
    device = MODEL._embed_tokens.weight.device
    inputs = {k: (v.to(device) if hasattr(v, "to") else v) for k, v in inputs.items()}
    input_ids = inputs["input_ids"]
    seq_len = int(input_ids.shape[1])
    position = CAP["position"]

    final_layer = MODEL.n_layers - 1
    record_at = sorted(set(CAP["layers"]) | {final_layer})
    with ActivationRecorder(MODEL.layers, at=record_at) as recorder:
        MODEL._hf_model(**inputs)
        residuals = {i: recorder.activations[i][0].detach() for i in record_at}

    h_final = residuals[final_layer][position].float()
    model_logits_pre, model_logits_capped = MODEL.unembed_pair(h_final)
    model_probs = torch.softmax(model_logits_capped.float(), dim=-1)
    top = model_logits_capped.float().topk(CAP["top_k"])
    model_topk = [
        {"token_id": int(i), "token": MODEL.tokenizer.decode([int(i)]),
         "logit": float(v), "prob": float(model_probs[int(i)])}
        for v, i in zip(top.values, top.indices)
    ]

    records = {"layers": {}}
    for layer in CAP["layers"]:
        h = residuals[layer][position].float()
        J = LENS.jacobians[layer].to(device=h.device, dtype=torch.float32)
        jlens_logits = MODEL.unembed_presoftcap(J @ h).float()
        jl_top = jlens_logits.topk(CAP["top_k"])
        jlens_topk = [
            {"token_id": int(i), "token": MODEL.tokenizer.decode([int(i)]),
             "logit": float(v), "prob": None}
            for v, i in zip(jl_top.values, jl_top.indices)
        ]
        rank = int((jlens_logits > jlens_logits[model_topk[0]["token_id"]]).sum())
        overlap = len({t["token_id"] for t in jlens_topk}
                      & {t["token_id"] for t in model_topk}) / CAP["top_k"]

        dictionary = JSpaceDictionary.from_lens(
            LENS, layer, MODEL._lm_head.weight, device=device,
            build_chunk_rows=16384,
        )
        result = gradient_pursuit(h.unsqueeze(0), dictionary, SETTINGS)
        pursuit_record = result.to_records()[0]
        labels = [MODEL.tokenizer.decode([t]) for t in pursuit_record["token_ids"]]
        cone = make_cone_record(
            pursuit_record,
            decoded_labels=labels,
            layer=layer,
            position=position,
            input_token_id=int(input_ids[0, position]),
            input_token=MODEL.tokenizer.decode([int(input_ids[0, position])]),
            prompt_hash=_hash16(prompt),
            prompt_slug=None,
            prompt_format="plain" if route["route"] == "processor_call" else "chat",
            run_provenance={
                "run_id": RUN_ID,
                "lens_fingerprint": LENS_VERIFICATION["file_sha256"],
                "model_revision": LOAD_INFO["model_revision"],
                "config_fingerprint": FINGERPRINT,
                "modality": modality,
                "exploratory_note": (
                    "text-fitted lens applied to a multimodal-conditioned "
                    "decoder state; exploratory, no modality-invariance claim"),
            },
        )
        records["layers"][layer] = {
            "jlens_topk": jlens_topk,
            "rank_of_model_top1": rank,
            "topk_overlap_with_model": overlap,
            "target_norm": float(h.norm()),
            "cone": cone,
        }
        del dictionary, result
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    modality_range = _token_range(
        input_ids.cpu(),
        INTERFACE["image_token_id"] if modality == "image_text"
        else INTERFACE["audio_token_id"])
    records.update({
        "schema": "jlens.multimodal.capture.v1",
        "modality": modality,
        "prompt": prompt,
        "prompt_hash": _hash16(prompt),
        "seq_len": seq_len,
        "position": position,
        "resolved_position": seq_len + position if position < 0 else position,
        "input_token_id": int(input_ids[0, position]),
        "input_token": MODEL.tokenizer.decode([int(input_ids[0, position])]),
        "model_topk": model_topk,
        "modality_token_range": modality_range,
        "processor_route": route,
        "processor_interface": INTERFACE,
        "asset_name": asset_name,
        "asset_meta": asset_meta or {},
        "readouts": {
            "model": "softcapped final logits (sampling pathway); probs from capped softmax",
            "jlens": "pre-softcap W_U norm(J h) (paper convention); rankings cap-invariant",
        },
    })
    return records


In [ ]:
# 10. IMAGE capture — one image, layer 38 (config), position -1.
IMAGE_RECORD = None
if MODEL is None:
    print("light path — skipping image capture.")
elif not IMAGE_PATH:
    print("IMAGE_PATH not set — skipping image capture.")
else:
    from PIL import Image
    image = Image.open(IMAGE_PATH).convert("RGB")
    asset_name = pathlib.Path(IMAGE_PATH).name
    IMAGE_RECORD = capture(
        IMAGE_PROMPT, modality="image_text", image=image,
        asset_name=asset_name,
        asset_meta={"width": image.width, "height": image.height,
                    "source_filename": asset_name},
    )
    shutil.copy2(IMAGE_PATH, OUTPUT_DIR / "assets" / asset_name)
    write_metadata(str(OUTPUT_DIR / "image_record.json"), IMAGE_RECORD)
    top1 = IMAGE_RECORD["model_topk"][0]
    print(f"image captured: seq_len={IMAGE_RECORD['seq_len']} "
          f"range={IMAGE_RECORD['modality_token_range']} "
          f"model top-1={top1['token']!r} ({top1['prob']:.3f})")
    for layer, rec in IMAGE_RECORD["layers"].items():
        print(f"  L{layer}: jlens rank_of_top1={rec['rank_of_model_top1']} "
              f"overlap={rec['topk_overlap_with_model']:.2f} "
              f"ef={rec['cone']['reconstruction']['explained_fraction']:.4f}")


In [ ]:
# 11. AUDIO capture — one clip, layer 38 (config), position -1. Loads with
# soundfile (16-bit PCM WAV recommended); mono-ized; the processor's own
# resampling/validation applies. Unsupported audio fails clearly above.
AUDIO_RECORD = None
if MODEL is None:
    print("light path — skipping audio capture.")
elif not AUDIO_PATH:
    print("AUDIO_PATH not set — skipping audio capture.")
else:
    try:
        import soundfile as sf
    except ImportError as exc:
        raise RuntimeError("pip install soundfile to load audio clips") from exc
    waveform, sample_rate = sf.read(AUDIO_PATH, dtype="float32")
    if waveform.ndim > 1:
        waveform = waveform.mean(axis=1)
    duration = len(waveform) / sample_rate
    asset_name = pathlib.Path(AUDIO_PATH).name
    AUDIO_RECORD = capture(
        AUDIO_PROMPT, modality="audio_text", audio=waveform,
        sampling_rate=sample_rate, asset_name=asset_name,
        asset_meta={"duration_seconds": duration, "sample_rate": int(sample_rate),
                    "source_filename": asset_name},
    )
    shutil.copy2(AUDIO_PATH, OUTPUT_DIR / "assets" / asset_name)
    write_metadata(str(OUTPUT_DIR / "audio_record.json"), AUDIO_RECORD)
    top1 = AUDIO_RECORD["model_topk"][0]
    print(f"audio captured: seq_len={AUDIO_RECORD['seq_len']} "
          f"duration={duration:.2f}s range={AUDIO_RECORD['modality_token_range']} "
          f"model top-1={top1['token']!r} ({top1['prob']:.3f})")
    for layer, rec in AUDIO_RECORD["layers"].items():
        print(f"  L{layer}: jlens rank_of_top1={rec['rank_of_model_top1']} "
              f"overlap={rec['topk_overlap_with_model']:.2f} "
              f"ef={rec['cone']['reconstruction']['explained_fraction']:.4f}")


In [ ]:
# 12. Assemble the explorer bundle (loads directly in the frontend as
# explorer/public/data/measured/multimodal.json; copy artifacts/assets/*
# alongside it as explorer/public/data/measured/assets/).
from datetime import datetime, timezone

from jlens.explorer_export import assemble_bundle, make_provenance, write_bundle

if MODEL is None:
    print("light path — skipping bundle assembly.")
elif IMAGE_RECORD is None and AUDIO_RECORD is None:
    print("nothing captured — no bundle to assemble.")
else:
    examples, layer_records, cones, traces = [], [], [], []
    for record in (IMAGE_RECORD, AUDIO_RECORD):
        if record is None:
            continue
        modality = record["modality"]
        stem = _slugify(pathlib.Path(record["asset_name"]).stem)
        slug = f"{modality.split('_')[0]}-{stem}"
        eid = make_example_id(modality, slug, record["prompt_hash"])
        asset_url = f"data/measured/assets/{record['asset_name']}"
        meta = record["asset_meta"]
        image_input = None
        audio_input = None
        if modality == "image_text":
            image_input = {
                "asset_url": asset_url,
                "width": meta.get("width"), "height": meta.get("height"),
                "prompt_text": record["prompt"],
                "modality_token_range": record["modality_token_range"],
                "processor_metadata": {
                    "route": record["processor_route"],
                    "processor_class": record["processor_interface"]["processor_class"],
                },
            }
        else:
            audio_input = {
                "asset_url": asset_url,
                "duration_seconds": meta.get("duration_seconds"),
                "sample_rate": meta.get("sample_rate"),
                "prompt_text": record["prompt"],
                "modality_token_range": record["modality_token_range"],
                "processor_metadata": {
                    "route": record["processor_route"],
                    "processor_class": record["processor_interface"]["processor_class"],
                },
            }
        position = record["position"]
        example = {
            "example_id": eid,
            "prompt_slug": slug,
            "prompt_hash": record["prompt_hash"],
            "category": "multimodal-exploratory",
            "format": "chat" if record["processor_route"]["route"] == "chat_template" else "plain",
            "modality": modality,
            "display_title": record["prompt"],
            "prompt_text": record["prompt"],
            "data_status": "measured",
            "seq_len": record["seq_len"],
            "selected_positions": [position],
            "model_output": {str(position): {
                "input_token_id": record["input_token_id"],
                "input_token": record["input_token"],
                "model_top1_id": record["model_topk"][0]["token_id"],
                "model_top1_token": record["model_topk"][0]["token"],
                "model_topk": record["model_topk"],
            }},
            "strength": None,
            "selection_reason": (
                "first exploratory application of the text-fitted lens to a "
                f"{modality} decoder state"),
            "input": {
                "text": {
                    "token_ids": None, "token_labels": None,
                    "positions_available": [position],
                    "special_token_flags": None,
                    "prompt_text_is_pre_template": False,
                    "tokenization_available": False,
                },
                "image": image_input,
                "audio": audio_input,
            },
        }
        examples.append(example)
        for layer, rec in record["layers"].items():
            layer_records.append({
                "example_id": eid, "layer": int(layer), "position": position,
                "source_site": "block_output", "data_status": "measured",
                "input_token_id": record["input_token_id"],
                "input_token": record["input_token"],
                "model_topk": record["model_topk"],
                "jlens_topk": rec["jlens_topk"],
                "rank_of_model_top1": rec["rank_of_model_top1"],
                "topk_overlap_with_model": rec["topk_overlap_with_model"],
                "eval_metadata": {"readouts": record["readouts"],
                                  "rank_convention": "0 = argmax"},
                "target_activation_norm": rec["cone"]["reconstruction"]["target_norm"],
                "residual_norm": rec["cone"]["reconstruction"]["residual_norm"],
                "relative_residual": rec["cone"]["reconstruction"]["relative_residual"],
                "explained_fraction": rec["cone"]["reconstruction"]["explained_fraction"],
            })
            cone = dict(rec["cone"])
            cone["_source_artifact"] = f"artifacts/{modality.split('_')[0]}_record.json"
            cone["_source_index"] = 0
            cones.append(_cone_to_bundle_record(
                cone, example=example, run_id=RUN_ID, atom_frequencies={}))
            traces.append(_cone_to_pursuit_trace(cone, example=example))

    provenance = make_provenance(
        source_run_ids=[RUN_ID],
        model_repo_id=LOAD_INFO["model_repo_id"],
        model_revision=LOAD_INFO["model_revision"],
        created_utc=datetime.now(timezone.utc).isoformat(timespec="seconds"),
        data_status="measured",
        modalities_present=sorted({e["modality"] for e in examples}),
        lens_fingerprint=LENS_VERIFICATION["file_sha256"],
        implementation_commit=ENV.get("local_commit"),
        notes=("Exploratory application of the text-fitted pilot lens to "
               "multimodal-conditioned decoder states; see docs/multimodal_capture.md."),
    )
    bundle = assemble_bundle(
        provenance=provenance, examples=examples, layer_records=layer_records,
        cones=cones, pursuit_traces=traces,
    )
    try:
        import jsonschema  # noqa: F401
        sha = write_bundle(bundle, str(OUTPUT_DIR / "multimodal_explorer_bundle.json"),
                           schema_path="schemas/explorer_bundle.schema.json")
    except ImportError:
        print("jsonschema not installed — writing without schema validation")
        sha = write_bundle(bundle, str(OUTPUT_DIR / "multimodal_explorer_bundle.json"))
    print(f"explorer bundle: {sha} ({len(examples)} examples)")


In [ ]:
# 13. Run manifest + human-readable summary.
if MODEL is None:
    print("model not loaded — nothing to summarize (light path).")
else:
    manifest = {
        "run_id": RUN_ID,
        "run_dir": str(RUN_DIR),
        "mode": "multimodal_capture",
        "config": CONFIG,
        "config_fingerprint": FINGERPRINT,
        "execution": EXECUTION,
        "lens_verification": LENS_VERIFICATION,
        "load_info": LOAD_INFO,
        "architecture_report": ARCH_REPORT,
        "processor_interface": INTERFACE,
        "captured": {
            "image": None if IMAGE_RECORD is None else {
                "asset": IMAGE_RECORD["asset_name"],
                "seq_len": IMAGE_RECORD["seq_len"],
                "modality_token_range": IMAGE_RECORD["modality_token_range"],
            },
            "audio": None if AUDIO_RECORD is None else {
                "asset": AUDIO_RECORD["asset_name"],
                "seq_len": AUDIO_RECORD["seq_len"],
                "modality_token_range": AUDIO_RECORD["modality_token_range"],
            },
        },
        "environment": ENV,
        "notes": (
            "Text-fitted frozen lens applied to multimodal-conditioned decoder "
            "states — exploratory; no modality-invariance, pixel-attribution, "
            "or audio-span-attribution claims."),
    }
    write_metadata(str(RUN_DIR / "run_metadata.json"), manifest)

    lines = [
        f"# Run {RUN_ID}",
        "",
        "- mode: multimodal_capture (frozen text-fitted lens, exploratory)",
        f"- lens: {LENS_VERIFICATION['file_sha256']}",
        f"- model: {LOAD_INFO['model_repo_id']} @ {LOAD_INFO['model_revision']}",
        f"- layers: {CAP['layers']}; k={CAP['k']}; position {CAP['position']}",
        f"- image: {manifest['captured']['image']}",
        f"- audio: {manifest['captured']['audio']}",
        "- artifacts: image_record.json, audio_record.json, "
        "multimodal_explorer_bundle.json (+ assets/)",
        "",
        "Next: copy artifacts/multimodal_explorer_bundle.json to "
        "explorer/public/data/measured/multimodal.json and artifacts/assets/* to "
        "explorer/public/data/measured/assets/ (see docs/multimodal_capture.md).",
    ]
    with open(RUN_DIR / "summary.md", "w", encoding="utf-8") as fh:
        fh.write("\n".join(lines))
    print("\n".join(lines))


## Summary and limitations

- These are the explorer's first image- and audio-conditioned records: model
  top-10, J-lens top-10, k=10 cone, and pursuit residual history at layer 38,
  position −1, for one example per modality.
- **The lens was fitted on text.** These records probe what the text-fitted
  lens reads out of multimodal-conditioned decoder states — they do not
  establish shared text/image/audio concepts. Comparisons across modalities
  are exploratory observations about *this* lens, not properties of the model.
- No pixel or audio-span attribution exists here: only the processor-exposed
  modality token ranges are recorded, and only when contiguous.
- Layer 35 capture: after a successful layer-38 run, set `capture.layers:
  [35, 38]` in the config and rerun (a fresh run directory is created).
- After the run: copy the bundle and assets into
  `explorer/public/data/measured/` — the explorer prefers measured bundles
  and stops offering the multimodal fixture.